# Agentic Workflow A

```
MAIN FLOW                           TOOLS

User Prompt
    │
    ▼
┌───────────────┐             ┌───────────────┐
│ News Gathering│────────────►│    SearXNG    │
│     Agent     │◄────────────│               │
└───────────────┘             └───────────────┘
    │
    ▼
┌───────────────┐             ┌───────────────┐
│ News Sentiment│────────────►│    FinBERT    │
│     Agent     │◄────────────│ Sentiment Tool│
└───────────────┘             └───────────────┘
    │
    ▼
┌────────────────┐
│ News Summarizer│
│     Agent      │
└────────────────┘
    │
    ▼
┌────────────────┐
│ News Reporting │
│     Agent      │
└────────────────┘
    │
    ▼
┌─────────────────┐
│   Structured    │
│ Financial Report│
└─────────────────┘
```

## Import Libraries

In [1]:
import torch
import wandb
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from crewai import Agent, Task, Crew, Process
from tqdm import tqdm
import os

/home/horizontal_roterien_katze/.local/share/mamba/envs/thesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

### Project Config

In [2]:
from pathlib import Path
import sys

# Project root
PROJECT_PATH = Path("..").resolve()

# Add project root to Python path
sys.path.append(str(PROJECT_PATH))

# Internal tools
from utils.tools.search_tool import search_searxng
from utils.tools.sentiment_tool import analyze_sentiment
from utils.tools.markdown_render_tool import (
    markdown_render_tool,
    render_markdown_report,
    REPORTS_PATH,
)

reports_path = str(REPORTS_PATH)

# Environment variables
from dotenv import load_dotenv

load_dotenv(PROJECT_PATH / ".env")

# LLM
from crewai import LLM

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14791.74it/s]


### LLM Config

In [3]:
llm = LLM(
    model="openai/gpt-5-mini"
)

## Tools

In [4]:
from crewai.tools import tool

search_tool = search_searxng

@tool("FinBERT Sentiment Analysis")
def sentiment_tool(text: str, model_type: str = "bert") -> str:
    """Analyze the sentiment of a news article text using FinBERT or BERT. Returns label and confidence score."""
    return str(analyze_sentiment(text, model_type))

markdown_render_tool = markdown_render_tool

## Create Agents

In [5]:
news_gathering = Agent(
    role="news_gathering",
    goal="Provide news latest articles about stock market and finance prompted by the user. Use the search tool to gather relevant information provide 5 until 10 news articles.",
    backstory="You are a news gathering agent that collects the latest news articles about stock market and finance based on user prompts. You will provide relevant and up-to-date information to assist users in making informed decisions.",
    tools=[search_tool],
    llm=llm,
)

In [6]:
news_sentiment = Agent(
    role="news_sentiment",
    goal="Use the sentiment tool to analyze the sentiment of news articles about stock market and finance prompted by the user. Provide a summary of the sentiment analysis results. the tool would output a sentiment of neutral, positive, or negative.",
    backstory="You are a sentiment analysis agent that analyzes the sentiment of news articles about stock market and finance based on user prompts. You will provide a summary of the sentiment analysis results to assist users in understanding the overall sentiment of the news articles.",
    tools=[sentiment_tool],
    llm=llm,
)

In [7]:
news_summarizing = Agent(
    role="news_summarizing",
    goal="summarize the news articles with the sentiment analysis results and provide a summary of the news articles about stock market and finance prompted by the user.",
    backstory="You are a news gathering agent that collects the latest news articles about stock market and finance based on user prompts. You will provide relevant and up-to-date information to assist users in making informed decisions.",
    llm=llm,
)

In [8]:
news_reporting = Agent(
    role="news_reporting",
    goal="Compile the summarized news articles with sentiment analysis results and provide a final report of the news articles about stock market and finance prompted by the user.",
    backstory="You are a news gathering agent that collects the latest news articles about stock market and finance based on user prompts. You will provide relevant and up-to-date information to assist users in making informed decisions.",
    tools=[markdown_render_tool],
    llm=llm,
)

## Create Tasks

In [9]:
task_gathering = Task(
    description=(
        "Search for the latest news articles about '{topic}' using the search tool. "
        "Collect relevant article titles, URLs, and content snippets worth analyzing."
    ),
    expected_output=(
        "A structured list of the latest relevant news articles about '{topic}', "
        "including title, URL, and content for each article."
    ),
    agent=news_gathering,
)

In [10]:
task_sentiment = Task(
    description=(
        "For each news article gathered about '{topic}', call the analyze_sentiment function "
        "(imported from utils.tools.sentiment_tool) on its text using model_type='finbert'. "
        "Aggregate the results into an overall sentiment summary."
    ),
    expected_output=(
        "A per-article sentiment breakdown (positive/negative/neutral with confidence) "
        "and an overall aggregated sentiment summary for '{topic}'."
    ),
    agent=news_sentiment,
    context=[task_gathering],
)

In [11]:
task_summarizing = Task(
    description=(
        "Summarize the gathered news articles about '{topic}' together with their sentiment analysis results "
        "into a clear, concise summary."
    ),
    expected_output=(
        "A well-structured summary of the latest news about '{topic}' that integrates "
        "the sentiment analysis findings."
    ),
    agent=news_summarizing,
    context=[task_gathering, task_sentiment],
)

In [12]:
task_reporting = Task(
    description=(
        "Compile the summarized news and sentiment results about '{topic}' into a final structured financial report, "
        "then use the markdown_render_tool (topic, summary, sentiment, overview) to save it as a markdown file. "
        "The tool always saves to the project's reports directory — do not pass a path in the filename."
    ),
    expected_output=(
        "A complete structured financial report about '{topic}' in markdown format, saved to disk."
    ),
    agent=news_reporting,
    context=[task_summarizing],
    output_file=os.path.join(reports_path, "financial_report.md"),
)

## Build Crew

In [13]:
crew = Crew(
    agents=[news_gathering, news_sentiment, news_summarizing, news_reporting],
    tasks=[task_gathering, task_sentiment, task_summarizing, task_reporting],
    process=Process.sequential,
    verbose=True,
)

## Run Workflow

In [14]:
topic = "NVIDIA stock"  # change to any stock / finance topic

result = await crew.kickoff_async(inputs={"topic": topic})

print(result)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.22                                                                                       │
│  Latest version:  1.15.23                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 93ba7a66-9048-45f3-a063-aa2c45f0cd8a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search for the latest news articles about 'NVIDIA stock' using the search tool. Collect relevant         │
│  article titles, URLs, and content snippets worth analyzing.                                                    │
│  ID: 70d3487e-46c6-4da7-83a2-bcab1ed078b3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: news_gathering                                                                                          │
│                                                                                                                 │
│  Task: Search for the latest news articles about 'NVIDIA stock' using the search tool. Collect relevant         │
│  article titles, URLs, and content snippets worth analyzing.                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: sear_xng_search                                                                                          │
│  Args: {'query': 'NVIDIA stock latest news 2026 October 2026 NVIDIA stock news'}                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: news_reporting                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I saved the requested report to disk as "NVIDIA_stock_report.md" and below is the full content of the          │
│  markdown file.                                                                                                 │
│                                                                                                                 │
│  Topic: NVIDIA stock                                                                                            │
│                                                                                                                 │
│  Top-line synthesis                                                                                             │
│                                                                                                                 │
│  - Market-moving items: Nvidia reported very strong Q2 FY2027 results (revenue $96.2B) and announced a record   │
│  $150B increase to its buyback authorization. The stock has rallied to fresh highs (near a ~$6T market cap      │
│  intraday), while valuation multiples have compressed to low levels versus recent history. Separately,          │
│  Bloomberg reported potential compliance/regulatory risk tied to chips reaching China despite export controls.  │
│  - Overall sentiment (FinBERT on the provided snippets/summaries): mostly neutral (6 of 8 articles), with one   │
│  strongly negative and one moderately positive signal. The weighted signal leans slightly negative because the  │
│  negative label carried very high model confidence — indicating balanced factual coverage but some important    │
│  risk-oriented or model-sensitive phrasing.                                                                     │
│  - Practical view: The news mix is fundamentally positive on demand and buyback support, but investors are      │
│  weighing valuation, cyclicality, and regulatory/compliance risks — likely producing the neutral/balanced tone  │
│  across coverage.                                                                                               │
│                                                                                                                 │
│  Key takeaways                                                                                                  │
│                                                                                                                 │
│  - Corporate results and buyback: Q2 revenue/profit beat and an unprecedented $150B share-repurchase addition   │
│  underline management confidence and are supportive for the stock.                                              │
│  - Market reaction/valuation: Shares hit intraday records and rallied; yet forward multiples have fallen        │
│  (reported ~<17x forward profit in one article), prompting both bullish arguments (stock still “cheap” to       │
│  some) and caution about multiple compression.                                                                  │
│  - Sector context: Nvidia’s outlook for extended AI-driven datacenter spend helped lift chip peers and sparked  │
│  a sector rally.                                                                                                │
│  - Risk/regulatory angle: Investigative reporting suggests Nvidia-designed chips have moved into China despite  │
│  U.S. controls, raising potential compliance, reputatio

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Compile the summarized news and sentiment results about 'NVIDIA stock' into a final structured           │
│  financial report, then use the markdown_render_tool (topic, summary, sentiment, overview) to save it as a      │
│  markdown file. The tool always saves to the project's reports directory — do not pass a path in the filename.  │
│  Agent: news_reporting                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

I saved the requested report to disk as "NVIDIA_stock_report.md" and below is the full content of the markdown file.

Topic: NVIDIA stock

Top-line synthesis

- Market-moving items: Nvidia reported very strong Q2 FY2027 results (revenue $96.2B) and announced a record $150B increase to its buyback authorization. The stock has rallied to fresh highs (near a ~$6T market cap intraday), while valuation multiples have compressed to low levels versus recent history. Separately, Bloomberg reported potential compliance/regulatory risk tied to chips reaching China despite export controls.
- Overall sentiment (FinBERT on the provided snippets/summaries): mostly neutral (6 of 8 articles), with one strongly negative and one moderately positive signal. The weighted signal leans slightly negative because the negative label carried very high model confidence — indicating balanced factual coverage but some important risk-oriented or model-sensitive phrasing.
- Practical view: The news mix is fundamenta

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 93ba7a66-9048-45f3-a063-aa2c45f0cd8a                                                                       │
│  Final Output: I saved the requested report to disk as "NVIDIA_stock_report.md" and below is the full content   │
│  of the markdown file.                                                                                          │
│                                                                                                                 │
│  Topic: NVIDIA stock                                                                                            │
│                                                                                                                 │
│  Top-line synthesis                                                                                             │
│                                                                                                                 │
│  - Market-moving items: Nvidia reported very strong Q2 FY2027 results (revenue $96.2B) and announced a record   │
│  $150B increase to its buyback authorization. The stock has rallied to fresh highs (near a ~$6T market cap      │
│  intraday), while valuation multiples have compressed to low levels versus recent history. Separately,          │
│  Bloomberg reported potential compliance/regulatory risk tied to chips reaching China despite export controls.  │
│  - Overall sentiment (FinBERT on the provided snippets/summaries): mostly neutral (6 of 8 articles), with one   │
│  strongly negative and one moderately positive signal. The weighted signal leans slightly negative because the  │
│  negative label carried very high model confidence — indicating balanced factual coverage but some important    │
│  risk-oriented or model-sensitive phrasing.                                                                     │
│  - Practical view: The news mix is fundamentally positive on demand and buyback support, but investors are      │
│  weighing valuation, cyclicality, and regulatory/compliance risks — likely producing the neutral/balanced tone  │
│  across coverage.                                                                                               │
│                                                                                                                 │
│  Key takeaways                                                                                                  │
│                                                                                                                 │
│  - Corporate results and buyback: Q2 revenue/profit beat and an unprecedented $150B share-repurchase addition   │
│  underline management confidence and are supportive for the stock.                                              │
│  - Market reaction/valuation: Shares hit intraday records and rallied; yet forward multiples have fallen        │
│  (reported ~<17x forward profit in one article), prompting both bullish arguments (stock still “cheap” to       │
│  some) and caution about multiple compression.                                                                  │
│  - Sector context: Nvidia’s outlook for extended AI-driven datacenter spend helped lift chip peers and sparked  │
│  a sector rally.                                                                                                │
│  - Risk/regulatory angle: Investigative reporting suggests Nvidia-designed chips have moved into China despite  │
│  U.S. controls, raising potential compliance, reputati

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯